# 03 — Extreme Day Error Analysis

This is the core finding of the project: does the calendar-only baseline
actually perform worse on extreme-temperature days? This notebook proves it
(or doesn't — either way, the number is the point).

Run notebook 02 first so `data/processed/baseline_test_predictions.parquet`
exists.

In [1]:
# lets src/ be imported when running this notebook from the notebooks/ folder
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))


In [2]:
import pandas as pd

from src import config, utils

df = pd.read_parquet(config.JOINED_DATA_PATH).asfreq("h")
train, test = utils.time_ordered_split(df)

train: 2019-01-01 to 2024-12-31 (52608 rows)
test:  2025-01-01 to 2025-12-31 (8760 rows)


## Extreme-day thresholds — computed on training data only

This is the leakage-safe step. The percentile cutoff is calculated from
training-set temperatures only, then applied as a fixed number to the test
set. Computing this on the full range (including test data) would leak
future information into what counts as "extreme."

In [3]:
train_daily_temp = utils.daily_min_max_temp(train)
thresholds = utils.compute_extreme_thresholds(train_daily_temp)
print(thresholds)

{'low_cutoff': nan, 'high_cutoff': nan}


In [4]:
test_daily_temp = utils.daily_min_max_temp(test)
extreme_flags = utils.flag_extreme_days(test_daily_temp, thresholds)
extreme_dates = set(extreme_flags[extreme_flags].index.date)

print(f"{len(extreme_dates)} extreme days out of {len(test_daily_temp)} in the test year")

0 extreme days out of 365 in the test year


**If that count looks small (rough rule of thumb: under ~15-20 days),** the
segmented metrics below will be noisy. Widen `EXTREME_TEMP_PERCENTILE` in
`src/config.py` from 5 to 10 and rerun this notebook from the top, rather
than reporting a number computed on a handful of days.

## Baseline model — normal days vs extreme days

In [5]:
baseline_preds = pd.read_parquet(config.BASELINE_PREDICTIONS_PATH)
baseline_result = utils.segment_metrics_by_extreme_day(baseline_preds, extreme_dates)
baseline_result

d:\1_My Folders\Career\DataField\0_Projects\Data Analyst\03_Time-Series\ercot-forecasting\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\1_My Folders\Career\DataField\0_Projects\Data Analyst\03_Time-Series\ercot-forecasting\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


{'normal_mape': 67.37837227804776,
 'normal_rmse': 10747.413505715771,
 'extreme_mape': nan,
 'extreme_rmse': nan,
 'extreme_day_count': 0,
 'normal_day_count': 365}

This is the number that justifies the whole project: if `extreme_mape` is
meaningfully higher than `normal_mape`, that confirms the weather-blind model
really does struggle on extreme-temperature days. If it's roughly the same,
that's a real result too, worth being upfront about rather than downplaying —
say so plainly in the README rather than picking a number that fits the
narrative.